# GFN2 fine-tune of MACE-OFF on 250 rotaxane frames (Colab)

Runs the same workflow as the 10-frame macOS probe, at 250 frames and on a GPU:

1. **GFN2-xTB labels** for `data/rot1_sampled_250.xyz` (torch-free subprocess)
2. **Reference pool** for the latent-OOD detector, built from the MACE-OFF23 test split with *stock* `off-medium`
3. **Baseline (before)**: per-frame molecule-mean OOD for all 250 frames
4. **Fine-tune** `off-medium` on the GFN2 labels (energy **and** forces — the probe showed an energy-only hammer shifts the whole latent manifold)
5. **After**: rebuild the pool *through the finetuned encoder* (latent spaces are not comparable across checkpoints) and re-score
6. **Per-atom OOD maps** before/after for one frame

**First: Runtime → Change runtime type → GPU.** Total wall time ≈ 60–90 min.
If the pip cell changes `torch`/`numpy` versions, Runtime → Restart session, then re-run it (the clone is skipped on re-run).

In [ ]:
# @title 1. Clone the repo + install the stack {display-mode:"form"}
import os, subprocess, torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU attached — the fine-tune cell will be ~10x slower.")

if not os.path.exists("/content/mace/code/mace_calc.py"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/MauricioCafiero/MACE_UseAndTrain.git",
                    "/content/mace"], check=True)
    print("repo cloned")
%cd /content/mace
!pip -q install mace-torch tblite 2>&1 | tail -3
print("install done")

### What the probe taught us (carried into the defaults below)

On the 10-frame probe (energy-only fine-tune, 50 epochs), the unusual-chemistry score went **UP** ~3×: mean OOD 0.150 → 0.455 vs the original pool, and 0.256 even vs a pool rebuilt through the finetuned encoder. Decomposition: ~+0.20 wholesale latent-space shift + ~+0.10 genuine novelty — under an energy-dominated hammer the model moves off the OFF23 manifold entirely.

So this run changes three things: **forces carry real weight** (`forces_weight=100`, not 1), **fewer epochs** (30, with EMA + plateau LR decay), and the **self-pool comparison** (finetuned model vs pool built through the finetuned encoder) is the headline number.

In [ ]:
# @title 2. GFN2-xTB labels for the 250 frames (~3 min) {display-mode:"form"}
# tblite and torch bundle separate OpenMP runtimes and segfault when loaded
# into one process, so labeling runs as its own torch-free worker pool
# (code/gfn2_label.py never imports torch).
!mkdir -p runs
!python code/gfn2_label.py data/rot1_sampled_250.xyz runs/rot250_gfn2.xyz --workers 4

In [ ]:
# @title 3. Split into train / valid (every 10th frame held out) {display-mode:"form"}
import sys; sys.path.insert(0, "code")
from ase.io import read, write

frames = read("runs/rot250_gfn2.xyz", index=":")
valid = [a for i, a in enumerate(frames) if i % 10 == 0]
train = [a for i, a in enumerate(frames) if i % 10 != 0]
write("runs/rot250_gfn2_train.xyz", train, format="extxyz")
write("runs/rot250_gfn2_valid.xyz", valid, format="extxyz")
print(f"{len(train)} train / {len(valid)} valid frames")

In [ ]:
# @title 4. Build the stock off-medium reference pool (one-time, ~15 min) {display-mode:"form"}
# Downloads the MACE-OFF23 test split (~81 MB) and encodes 2500 drug-like
# frames into the latent pool that trust.py / ood_map.py score against.
import sys; sys.path.insert(0, "code")
from activation_ood import ReferencePool

pool = ReferencePool.build(n_frames=2500, out="data/off23_pool.npz")
print("pool atoms:", pool.atom_vecs.shape[0])

In [ ]:
# @title 5. Baseline: score all 250 frames with stock off-medium ("before", ~2 min) {display-mode:"form"}
import numpy as np
import mace_calc as mc
from trust_frames import read_frames
from activation_ood import atom_ood_scores

frames = read_frames("data/rot1_sampled_250.xyz")
mc.attach(frames[0], model="off-medium", device="cuda", dtype="float32")
shared = frames[0].calc            # one calculator shared across all frames

before = []
for k, at in enumerate(frames):
    at.calc = shared
    d = atom_ood_scores(at, pool)["distances"]
    before.append(float(np.nanmean(d)))
    if k % 50 == 0:
        print(f"frame {k:3d}  mean OOD {before[-1]:.3f}")
before = np.array(before)
print(f"\nSTOCK off-medium: mean {before.mean():.3f}  "
      f"range [{before.min():.3f}, {before.max():.3f}]")

In [ ]:
# @title 6. Fine-tune off-medium on the GFN2 labels (GPU, ~30–60 min) {display-mode:"form"}
from finetune_mace import run_finetune, find_latest_model

run_finetune(
    "runs/rot250_gfn2_train.xyz", "runs/rot250_gfn2_valid.xyz",
    foundation_model="off-medium", name="rot250", results_dir="runs",
    max_num_epochs=30,
    energy_weight=100.0, forces_weight=100.0,
    scheduler="ReduceLROnPlateau",
    device="cuda", default_dtype="float32",
    extra=("--batch_size=8", "--valid_batch_size=8", "--eval_interval=2"),
)
model_path = find_latest_model("runs", "rot250")
print("finetuned checkpoint:", model_path)

In [ ]:
# @title 7. Rebuild the pool through the finetuned encoder + re-score ("after") {display-mode:"form"}
# Latent spaces are not comparable across checkpoints, so the finetuned model
# is scored against a pool built through the SAME encoder (the stage-5 control
# from the probe). mc.get_calculator is monkeypatched to hand out the
# finetuned calculator everywhere a pool build asks for one.
from pathlib import Path
from mace.calculators import MACECalculator
import mace_calc as mc
import matplotlib.pyplot as plt

ft = MACECalculator(model_paths=str(model_path), device="cuda",
                    default_dtype="float32")
mc.get_calculator = lambda **kw: ft   # pool build runs the finetuned encoder

ft_pool = ReferencePool.build(n_frames=2500, out="data/off23_pool_ft.npz")

after = []
for k, at in enumerate(frames):
    at.calc = ft
    after.append(float(np.nanmean(atom_ood_scores(at, ft_pool)["distances"])))
after = np.array(after)

plt.figure(figsize=(7.5, 4))
plt.hist(before, bins=30, alpha=0.55,
         label=f"stock off-medium / own pool  (mean {before.mean():.3f})")
plt.hist(after, bins=30, alpha=0.55,
         label=f"finetuned / own pool  (mean {after.mean():.3f})")
plt.axvline(0.25, ls="--", c="k", lw=1)
plt.text(0.255, plt.ylim()[1] * 0.9, "trust line (0.25)", fontsize=8)
plt.xlabel("molecule-mean latent OOD (cosine)")
plt.ylabel("frames")
plt.title("250 rotaxane frames: before vs after GFN2 fine-tune")
plt.legend(fontsize=8)
plt.show()

print(f"{'':<12}{'before':>8}{'after':>8}")
print(f"{'mean':<12}{before.mean():8.3f}{after.mean():8.3f}")
print(f"{'max':<12}{before.max():8.3f}{after.max():8.3f}")
print(f"{'frames>0.25':<12}{int((before > 0.25).sum()):8d}"
      f"{int((after > 0.25).sum()):8d}")

In [ ]:
# @title 8. Per-atom OOD maps, before vs after (frame 0) {display-mode:"form"}
# Before: the stock ood_map.py CLI (scores with off-medium + stock pool).
!python code/ood_map.py data/rot1_sampled_250.xyz --model off-medium --frame 0 --out-dir viz_before

# After: same tool in-process, pointed at the finetuned checkpoint + its pool.
from pathlib import Path
import trust
trust.POOLS["rot250-ft"] = Path("data/off23_pool_ft.npz")  # ood_map shares this dict
import ood_map
mc.get_calculator = lambda **kw: ft          # ood_map's mc is this same module
ood_map.main(["data/rot1_sampled_250.xyz", "--model", "rot250-ft",
              "--frame", "0", "--out-dir", "viz_after"])

from IPython.display import Image, display
print("\nBEFORE (stock):"), display(Image("viz_before/frame00_ood.png"))
print("AFTER (finetuned):"), display(Image("viz_after/frame00_ood.png"))

In [ ]:
# @title 9. Download the finetuned checkpoint {display-mode:"form"}
from google.colab import files
files.download(str(model_path))

### Reading the result

- **The headline number is cell 7's `after` mean** — finetuned model vs its own rebuilt pool, i.e. novelty *after* accounting for the wholesale latent-space shift. The probe's value there was 0.256 (still above the 0.15 stock baseline and the 0.25 trust line). With forces weighted properly and 25× more data, that is the number to beat.
- **Unusual ≠ unreliable.** The per-atom/localization score flags chemistry that is far from the OFF23 training distribution; the molecule-mean verdict (≤ 0.25 → TRUST) is the reliability pre-filter. Judge the fine-tune by the mean, look at the maps for *where*.
- **Scoring dtype.** Everything on Colab runs float32 for speed (the macOS probe used float64). Cosine distances shift at the ~0.001 level — far below the 0.25 threshold — but don't mix f32 scores with the f64 numbers in OOD_NOTES.md.
- **Knobs worth touching for a second run:** `max_num_epochs` (30 → 15–20 if the after-mean drifts up and validation loss plateaus early), `forces_weight` (100 → 1000 for an even stiffer PES), and the `i % 10` split ratio.
- **Parity check (optional):** absolute energies of stock MACE, the finetuned model, and GFN2 sit on different E0 baselines and must never be compared directly — only differences (ΔE between frames within one model) or held-out validation RMSE from the training log are meaningful.